# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [2]:
# importar librerías
import pandas as pd

In [3]:
# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv') 
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

In [4]:
# explorar datasets

print(orders.head())
print(orders.info())



  id_pedido id_usuario fecha_hora_pedido       pais dispositivo  \
0   order_0  user_6993        2025-05-22  Argentina     desktop   
1   order_1  user_1329        2025-06-15     Mexico     desktop   
2   order_2  user_3194        2025-05-02  Argentina     desktop   
3   order_3  user_4510        2025-06-09   Colombia      mobile   
4   order_4  user_5044        2025-03-30  Argentina     desktop   

  fuente_referencia       nombre_producto categoria_producto  cantidad  \
0           organic       Jacket-Winter-M               Moda       2.0   
1       paid_search  Tablet-Standard-64GB        Electronica       1.0   
2            social        Blender-XL-Red              Hogar       2.0   
3            social  Tablet-Standard-64GB        Electronica       1.0   
4       paid_search        Blender-XL-Red              Hogar       1.0   

   precio_unitario  monto_descuento  monto_total  
0           332.69              0.0       665.37  
1           176.86              5.0       171.86  

In [5]:
print(catalog.head())
print(catalog.info())


        nombre_producto categoria_producto  costo_unitario  \
0    Laptop-Gaming-16GB        Electrónica          280.68   
1       Phone-Pro-128GB        Electrónica           10.12   
2  Tablet-Standard-64GB        Electrónica           25.21   
3        Blender-XL-Red              Hogar          176.64   
4      Vacuum-Pro-Black              Hogar           16.60   

                 proveedor  
0   Fuller, Pena and Myers  
1                 King Ltd  
2               Bowers LLC  
3                Long-Reid  
4  Rivera, Carr and Finley  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes
No

In [6]:
print(marketing.head())
print(marketing.info())

        fecha      pais            id_campaña        canal    gasto
0  2025-01-01    Mexico        organic_Mexico      organic  2446.25
1  2025-01-01    Mexico    paid_search_Mexico  paid_search  2704.34
2  2025-01-01    Mexico         social_Mexico       social  2045.01
3  2025-01-01  Colombia      organic_Colombia      organic  2597.21
4  2025-01-01  Colombia  paid_search_Colombia  paid_search  1771.40
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB
None


---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---

In [7]:
# Hacer la revisión 
orders["fecha_hora_pedido"] = pd.to_datetime(orders["fecha_hora_pedido"],errors='coerce')
marketing["fecha"] = pd.to_datetime(marketing["fecha"], errors='coerce')
print(orders.info()) # Para revisar si se hizo el cambio
print(orders["fecha_hora_pedido"].isna().sum()) # Para revisar si alguno quedó como null
print(marketing.info())
print(marketing["fecha"].isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           25100 non-null  object        
 1   id_usuario          25100 non-null  object        
 2   fecha_hora_pedido   25100 non-null  datetime64[ns]
 3   pais                24800 non-null  object        
 4   dispositivo         25080 non-null  object        
 5   fuente_referencia   25070 non-null  object        
 6   nombre_producto     25070 non-null  object        
 7   categoria_producto  25020 non-null  object        
 8   cantidad            25050 non-null  float64       
 9   precio_unitario     25050 non-null  float64       
 10  monto_descuento     25050 non-null  float64       
 11  monto_total         25100 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(7)
memory usage: 2.3+ MB
None
0
<class 'pandas.core.frame.Dat

In [8]:
# Identificar columnas númericas en los 3 df
num_cols_orders = orders.select_dtypes(include="number").columns
num_cols_marketing = marketing.select_dtypes(include="number").columns
num_cols_catalog = catalog.select_dtypes(include="number").columns

# Función que revise los negativos y los ceros

def revisar_valores(df,num_cols):
    resultados = []

    for col in num_cols:
        negativos = (df[col] < 0).sum()
        ceros = (df[col] == 0).sum()

        resultados.append({
            'columna': col, 
            'negativos': negativos,
            'ceros': ceros
        })
    return pd.DataFrame(resultados)

# Instancias:
rev_orders = revisar_valores(orders, num_cols_orders)
rev_marketing = revisar_valores(marketing, num_cols_marketing)
rev_catalog = revisar_valores(catalog, num_cols_catalog)

# Ver resultados

print(rev_orders.sort_values(by='negativos', ascending = False))
print(rev_marketing.sort_values(by='negativos', ascending = False))
print(rev_catalog.sort_values(by='negativos', ascending = False))

           columna  negativos  ceros
0         cantidad          4      0
3      monto_total          4      0
1  precio_unitario          0      0
2  monto_descuento          0  12551
  columna  negativos  ceros
0   gasto          0      0
          columna  negativos  ceros
0  costo_unitario          0      0


In [9]:
orders[orders["cantidad"] < 0]

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
266,order_266,user_7011,2025-03-13,NaN,desktop,paid_search,Phone-Pro-128GB,Electronica,-2.0,101.31,10.0,-192.62
267,order_267,user_1087,2025-05-07,NaN,desktop,social,Phone-Pro-128GB,Electronica,-1.0,43.50,5.0,-38.50
268,order_268,user_84,2025-02-19,NaN,desktop,organic,Phone-Pro-128GB,Electronica,-1.0,497.65,5.0,-492.65
269,order_269,user_3323,2025-05-25,NaN,desktop,paid_search,Phone-Pro-128GB,Electronica,-1.0,423.53,0.0,-423.53


In [10]:
orders[orders["id_usuario"] == "user_7011"] # Para revisar si los negativos son devoluciones. Parece que no. 

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
266,order_266,user_7011,2025-03-13,NaN,desktop,paid_search,Phone-Pro-128GB,Electronica,-2.0,101.31,10.0,-192.62
10343,order_10343,user_7011,2025-03-14,Colombia,desktop,organic,Phone-Pro-128GB,Electronica,2.0,356.83,0.0,713.67


In [11]:
# Se revisarn las filas duplicadas
print(orders.duplicated().sum())
# Se revisan los id_pedido duplicados
print(orders['id_pedido'].duplicated().sum())

100
100


In [12]:
orders[orders["id_pedido"].duplicated(keep=False)].sort_values("id_pedido")

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
25023,order_10082,user_690,2025-02-17,Argentina,desktop,social,Sneakers-Urban-42,Moda,2.0,221.34,0.0,442.67
10082,order_10082,user_690,2025-02-17,Argentina,desktop,social,Sneakers-Urban-42,Moda,2.0,221.34,0.0,442.67
25037,order_10709,user_6783,2025-02-16,Colombia,mobile,organic,Jacket-Winter-M,Moda,1.0,170.10,10.0,160.10
10709,order_10709,user_6783,2025-02-16,Colombia,mobile,organic,Jacket-Winter-M,Moda,1.0,170.10,10.0,160.10
25065,order_10829,user_7697,2025-01-23,Argentina,mobile,social,Phone-Pro-128GB,Electronica,2.0,115.54,0.0,231.08
...,...,...,...,...,...,...,...,...,...,...,...,...
25048,order_8326,user_6177,2025-06-14,Argentina,mobile,organic,Vacuum-Pro-Black,Hogar,1.0,457.53,0.0,457.53
25043,order_8414,user_4073,2025-05-10,Colombia,mobile,social,Jacket-Winter-M,Moda,2.0,144.35,10.0,278.71
8414,order_8414,user_4073,2025-05-10,Colombia,mobile,social,Jacket-Winter-M,Moda,2.0,144.35,10.0,278.71
25056,order_974,user_3262,2025-05-04,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,2.0,268.58,5.0,532.16


In [13]:
##### Se eliminan los duplicados. ########
# Se revisó también: orders.groupby("id_pedido").size().value_counts() y orders.groupby("id_pedido").nunique()
orders = orders.drop_duplicates()

In [14]:
'''Se revisa la proporción de columna con datos nulos. En general son pocos, siendo la columna
de país la que presenta la mayor proporción. A fin de conservar la información, se imputa como desconocido''' 
orders.isna().mean().sort_values(ascending = False)

pais                  0.0120
categoria_producto    0.0032
cantidad              0.0020
precio_unitario       0.0020
monto_descuento       0.0020
fuente_referencia     0.0012
nombre_producto       0.0012
dispositivo           0.0008
id_pedido             0.0000
id_usuario            0.0000
fecha_hora_pedido     0.0000
monto_total           0.0000
dtype: float64

In [15]:
orders["id_pedido"].nunique(), len(orders)

(25000, 25000)

In [16]:
##### Imputación de Columnas categóricas. 
cols_cat = [
    "pais",
    "dispositivo",
    "fuente_referencia",
    "nombre_producto",
    "categoria_producto"
]

orders[cols_cat] = orders[cols_cat].fillna("Desc")

In [17]:
orders[cols_cat].info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 25000 entries, 0 to 24999
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   pais                25000 non-null  object
 1   dispositivo         25000 non-null  object
 2   fuente_referencia   25000 non-null  object
 3   nombre_producto     25000 non-null  object
 4   categoria_producto  25000 non-null  object
dtypes: object(5)
memory usage: 1.1+ MB


In [18]:
orders[cols_cat].describe()

,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto
count,25000,25000,25000,25000,25000
unique,7,3,4,8,4
top,Colombia,desktop,social,Blender-XL-Red,Hogar
freq,7481,12711,8402,4179,8346


In [19]:
orders["pais"].value_counts()

Colombia     7481
Mexico       7478
Argentina    7259
mexico        863
colombia      823
argentina     796
Desc          300
Name: pais, dtype: int64

In [20]:

orders["pais"] = orders["pais"].str.strip().str.title()
orders["pais"] = orders["pais"].replace({
    "Desconocido": "Unknown"
})

In [21]:
# orders["fuente_referencia"].value_counts()
# orders["dispositivo"].value_counts()
# orders["categoria_producto"].value_counts()
orders["pais"].value_counts()

Mexico       8341
Colombia     8304
Argentina    8055
Desc          300
Name: pais, dtype: int64

In [22]:
#### Revisar nulos en columnas numéricas. 
orders[["cantidad","precio_unitario","monto_descuento"]].isna().sum()

cantidad           50
precio_unitario    50
monto_descuento    50
dtype: int64

In [23]:
orders[orders["cantidad"].isna()]
'''Se identificaron 50 registros con información incompleta en variables de detalle (cantidad, precio), 
pero con monto total válido. En este punto son conservados y etiquetados para evitar pérdida de 
información en métricas agregadas'''

orders["flag_incompleto"] = orders[[
    "cantidad",
    "precio_unitario",
    "monto_descuento"
]].isna().any(axis=1)



In [24]:
'''El impacto es prácticamente nulo: 21628 / 51965834 = 0.04%'''

orders.groupby("flag_incompleto")["monto_total"].sum()



flag_incompleto
False    51965834.26
True        21628.00
Name: monto_total, dtype: float64

In [25]:
'''A fin de no afectar métricas, simplicar el dataset y evitar incosistencias, y dado que el impacto es
prácticamente nulo, se eliminan los registros que tienen bandera.''' 
orders = orders[orders["flag_incompleto"] == False]
print(orders.shape)
print(orders.isna().sum())
print(orders.describe())


(24950, 13)
id_pedido             0
id_usuario            0
fecha_hora_pedido     0
pais                  0
dispositivo           0
fuente_referencia     0
nombre_producto       0
categoria_producto    0
cantidad              0
precio_unitario       0
monto_descuento       0
monto_total           0
flag_incompleto       0
dtype: int64
           cantidad  precio_unitario  monto_descuento   monto_total
count  24950.000000     24950.000000     24950.000000  2.495000e+04
mean       7.115110       259.374527         4.502605  2.082799e+03
std      296.869961       138.690738         5.224057  9.924687e+04
min       -2.000000        20.030000         0.000000 -4.926500e+02
25%        1.000000       138.470000         0.000000  1.806150e+02
50%        2.000000       258.795000         0.000000  3.415600e+02
75%        2.000000       380.377500        10.000000  5.183475e+02
max    20000.000000       499.960000        15.000000  8.840200e+06


In [26]:

orders[orders["cantidad"] < 0]
orders = orders[orders["cantidad"] > 0] # Se eliminan los que tienen cantidades negativas.

In [27]:
'''Se revisarn los que tienen cantidades muy grandes.'''
'''El archivo parece contener tanto ventas minoristas (retail) como mayoristas (bulk), lo que podría
distorcionar algunos valores estadísticos. Tal vez sea recomendable que, más adelante, se haga una segmentación 
por estos tipos de venta.
'''
orders[orders["cantidad"] > 100]



,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,flag_incompleto
3521,order_3521,user_5812,2025-02-03,Mexico,mobile,paid_search,Laptop-Gaming-16GB,Electronica,10000.0,43.14,0.0,431400.0,False
3522,order_3522,user_3575,2025-03-29,Argentina,desktop,social,Laptop-Gaming-16GB,Electronica,10000.0,280.55,0.0,2805500.0,False
3586,order_3586,user_3380,2025-02-03,Mexico,mobile,paid_search,Laptop-Gaming-16GB,Electronica,10000.0,490.35,0.0,4903500.0,False
3643,order_3643,user_4440,2025-01-07,Colombia,desktop,social,Laptop-Gaming-16GB,Electronica,10000.0,238.15,0.0,2381500.0,False
3656,order_3656,user_884,2025-01-01,Argentina,mobile,organic,Laptop-Gaming-16GB,Electronica,20000.0,297.66,0.0,5953200.0,False
3668,order_3668,user_7270,2025-06-24,Mexico,mobile,paid_search,Laptop-Gaming-16GB,Electronica,20000.0,348.31,0.0,6966200.0,False
3689,order_3689,user_6566,2025-06-16,Mexico,desktop,paid_search,Laptop-Gaming-16GB,Electronica,10000.0,87.69,0.0,876900.0,False
3722,order_3722,user_4723,2025-05-09,Argentina,mobile,paid_search,Laptop-Gaming-16GB,Electronica,20000.0,442.01,0.0,8840200.0,False
3726,order_3726,user_2536,2025-02-12,Colombia,desktop,social,Laptop-Gaming-16GB,Electronica,20000.0,290.85,0.0,5817000.0,False
3748,order_3748,user_7096,2025-02-23,Mexico,desktop,organic,Laptop-Gaming-16GB,Electronica,10000.0,336.93,0.0,3369300.0,False


In [28]:
# Se vuelven a revisar los datos estadísticos. 
print(orders.describe())

           cantidad  precio_unitario  monto_descuento   monto_total
count  24946.000000     24946.000000     24946.000000  2.494600e+04
mean       7.116452       259.373385         4.502525  2.083179e+03
std      296.893743       138.679411         5.224280  9.925482e+04
min        1.000000        20.030000         0.000000  5.240000e+00
25%        1.000000       138.502500         0.000000  1.806400e+02
50%        2.000000       258.795000         0.000000  3.416050e+02
75%        2.000000       380.365000        10.000000  5.184375e+02
max    20000.000000       499.960000        15.000000  8.840200e+06


---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.


In [29]:


# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)



---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

In [30]:
# Determinación de intresos totales (Revenue)

revenue = orders['monto_total'].sum()
revenue

51966981.56

In [31]:
# Determinación del costo: 
catalog.head()
#Left join 
orders = orders.merge(catalog, on='nombre_producto', how="left")
# Se eliminan 30 filas que tienen costo unitario desconocido
orders = orders.dropna(subset=['costo_unitario'])
# Calcular los costos totales
orders['costo_total'] = orders['cantidad'] * orders['costo_unitario']
orders['costo_total'].sum()

43124069.010000005

In [32]:
# Determinación de la ganancia(utilidad) (profit)
orders['profit'] = orders['monto_total'] - orders['costo_total']
print(orders['profit'])
print(orders['profit'].sum())

0        286.75
1        146.65
2       -157.29
3        217.66
4        159.64
          ...  
24941     39.36
24942    537.61
24943     37.54
24944    801.70
24945     72.84
Name: profit, Length: 24916, dtype: float64
8830649.93


In [33]:
# Guardar como variables:
revenue = orders["monto_total"].sum()
cost = orders["costo_total"].sum()
profit = revenue - cost

In [34]:
# Determinar la inversión en marketing
total_marketing = marketing['gasto'].sum()
total_marketing

2871843.53

In [35]:
# Determinar el profit después de gastos de marketing
profit_neto = profit - total_marketing
profit_neto

5958806.399999993

In [36]:
# Calcular el margen real
profit_margen_real = profit_neto / revenue
profit_margen_real

0.11469230363620159

In [37]:
# ROI de marketing
# ROI = 2.07 (por cada $1 invertido, se genera $2.07 de profit bruto antes de marketing o $1.07 neto, según definición)
roi_marketing = (profit - total_marketing) / total_marketing
roi_marketing

2.0749063581468845

In [38]:
#### 2.2.1 Determinar el ticket promedio por orden
ticket_promedio = orders["monto_total"].sum() / orders["id_pedido"].nunique()
ticket_promedio

2085.195012843153

In [39]:
# Otra manera de hacerlo, para verificar. 
# El promedio puede estar muy afectado por las ventas de mayoreo
ticket_promedio2 = orders["monto_total"].mean()
ticket_promedio2

2085.195012843153

In [40]:
# 2.2.2 Cantidad de promedio de productos por orden
'''El promedio está fuertemente sesgado por las ventas de mayoreo, como ya se había observado 
con anterioridad, pues hay pedidos de 10k y 20k'''
cantidad_promedio_prod = (orders.groupby("id_pedido")["cantidad"].sum().mean())
cantidad_promedio_prod

7.123213999036763

In [41]:
# 2.2.3 Producto más vendido
'''El volumen de ventas se concentra en el producto Laptop-Gaming-16GB, por una diferencia muy considerable
Este producto es el que se suele encontrar en las ventas de mayoreo. Se podría decir que hay una dependencia
considerable de las ventas de un solo producto.'''
producto_mas_vendido = (
    orders.groupby("nombre_producto")["cantidad"]
    .sum()
    .sort_values(ascending=False)
)
producto_mas_vendido

nombre_producto
Laptop-Gaming-16GB      144198.0
Vacuum-Pro-Black          6284.0
Blender-XL-Red            6279.0
Jacket-Winter-M           6256.0
Sneakers-Urban-42         6172.0
Tablet-Standard-64GB      4153.0
Phone-Pro-128GB           4140.0
Name: cantidad, dtype: float64

In [42]:
# Gasto en marketing por canal
gasto_canal = marketing.groupby("canal")["gasto"].sum()
gasto_canal

canal
organic        913533.01
paid_search    863088.21
social         918043.21
Name: gasto, dtype: float64

In [43]:
# Determinar el ROI por canal 
'''paid_search es considerablemente más eficiente, siendo prácticamente el doble o más que los otros canales
Nuevamente, esto puede deberse al impacto de las ventas de mayoreo, por clientes de alto volumen.'''
revenue_canal = orders.groupby("fuente_referencia")["monto_total"].sum()
gasto_canal = marketing.groupby("canal")["gasto"].sum()

roi_canal = (revenue_canal - gasto_canal) / gasto_canal
roi_canal.sort_values(ascending=False)

fuente_referencia
paid_search    28.229318
social         14.496559
organic        12.683937
dtype: float64

---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [44]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [45]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [46]:
events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 8 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   id_usuario          120000 non-null  object
 1   id_sesion           120000 non-null  object
 2   nombre_evento       120000 non-null  object
 3   timestamp_evento    120000 non-null  object
 4   pais                120000 non-null  object
 5   dispositivo         120000 non-null  object
 6   fuente_referencia   120000 non-null  object
 7   categoria_producto  120000 non-null  object
dtypes: object(8)
memory usage: 7.3+ MB


In [47]:
events["nombre_evento"].value_counts()

first_visit         29957
add_to_cart         24157
select_item         23887
begin_checkout      17971
add_payment_info    12018
purchase            12010
Name: nombre_evento, dtype: int64

In [48]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
SELECT 
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS total_usuarios
FROM events
GROUP BY nombre_evento
ORDER BY total_usuarios DESC;
'''

totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,total_usuarios
0,first_visit,7796
1,add_to_cart,7634
2,select_item,7582
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [49]:
# Para ordenarlo según el orden del proceso 
query_totals = '''
SELECT 
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS total_usuarios
FROM events
GROUP BY nombre_evento
ORDER BY 
    CASE 
        WHEN nombre_evento = 'first_visit' THEN 1
        WHEN nombre_evento = 'select_item' THEN 2
        WHEN nombre_evento = 'add_to_cart' THEN 3
        WHEN nombre_evento = 'begin_checkout' THEN 4
        WHEN nombre_evento = 'add_payment_info' THEN 5
        WHEN nombre_evento = 'purchase' THEN 6
    END;
'''
totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,total_usuarios
0,first_visit,7796
1,select_item,7582
2,add_to_cart,7634
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [50]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH funnel AS (
    SELECT 
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios,
        CASE 
            WHEN nombre_evento = 'first_visit' THEN 1
            WHEN nombre_evento = 'select_item' THEN 2
            WHEN nombre_evento = 'add_to_cart' THEN 3
            WHEN nombre_evento = 'begin_checkout' THEN 4
            WHEN nombre_evento = 'add_payment_info' THEN 5
            WHEN nombre_evento = 'purchase' THEN 6
        END AS orden
    FROM events
    GROUP BY nombre_evento
)

SELECT 
    nombre_evento,
    usuarios,
    LAG(usuarios) OVER (ORDER BY orden) AS usuarios_previos,
    usuarios * 1.0 / LAG(usuarios) OVER (ORDER BY orden) AS tasa_conversion
FROM funnel
ORDER BY orden;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,nombre_evento,usuarios,usuarios_previos,tasa_conversion
0,first_visit,7796,NaN,NaN
1,select_item,7582,7796.0,0.972550
2,add_to_cart,7634,7582.0,1.006858
3,begin_checkout,7208,7634.0,0.944197
4,add_payment_info,6250,7208.0,0.867092
5,purchase,6240,6250.0,0.998400



La mayor caída se da entre el begin_checkout y el add_payment_info, lo que sugiere que por algún motivo se 
frenan antes de hacer la compra. Sin embargo sigue siendo un indicador que convierte bien, pues logra un 86.7%.  
Como dato adicional, hay más productos agregados que seleccionados, lo sugiere que no es un funnel totalmente lineal. 
Quiza los usuarios entran directamente al carrito o hay alguna otra manera de agregar productos sin seleccionarlos.
Se logra una conversión final (6240/7796 = 80%) muy alta, siendo un funnel bastante eficiente. 


---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [52]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [53]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [54]:
print(users.info())
print(user_activity.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id_usuario      8000 non-null   object
 1   fecha_registro  8000 non-null   object
 2   país            8000 non-null   object
 3   dispositivo     8000 non-null   object
 4   tipo_plan       8000 non-null   object
dtypes: object(5)
memory usage: 312.6+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32000 entries, 0 to 31999
Data columns (total 4 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   id_usuario             32000 non-null  object
 1   fecha_actividad        32000 non-null  object
 2   dias_despues_registro  32000 non-null  int64 
 3   activo                 32000 non-null  int64 
dtypes: int64(2), object(2)
memory usage: 1000.1+ KB
None


In [55]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH cohortes AS (
    SELECT
        id_usuario,
        CAST(fecha_registro AS DATE) AS fecha_registro,
        DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS cohorte_mes
    FROM users
),

actividad AS (
    SELECT
        id_usuario,
        dias_despues_registro,
        activo
    FROM user_activity
),

retencion AS (
    SELECT
        c.cohorte_mes,
        COUNT(DISTINCT c.id_usuario) AS usuarios_iniciales,

        COUNT(DISTINCT CASE 
            WHEN a.dias_despues_registro BETWEEN 1 AND 7 
                 AND a.activo = 1 
            THEN c.id_usuario END) AS retenido_w1,

        COUNT(DISTINCT CASE 
            WHEN a.dias_despues_registro BETWEEN 8 AND 14 
                 AND a.activo = 1 
            THEN c.id_usuario END) AS retenido_w2,

        COUNT(DISTINCT CASE 
            WHEN a.dias_despues_registro BETWEEN 15 AND 21 
                 AND a.activo = 1 
            THEN c.id_usuario END) AS retenido_w3

    FROM cohortes c
    LEFT JOIN actividad a
        ON c.id_usuario = a.id_usuario
    GROUP BY c.cohorte_mes
)

SELECT
    cohorte_mes,
    usuarios_iniciales,
    retenido_w1,
    retenido_w2,
    retenido_w3,

    retenido_w1 * 1.0 / usuarios_iniciales AS semana_1,
    retenido_w2 * 1.0 / usuarios_iniciales AS semana_2,
    retenido_w3 * 1.0 / usuarios_iniciales AS semana_3

FROM retencion
ORDER BY cohorte_mes;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte_mes,usuarios_iniciales,retenido_w1,retenido_w2,retenido_w3,semana_1,semana_2,semana_3
0,2025-01-01 00:00:00+00:00,1627,697,668,656,0.428396,0.410572,0.403196
1,2025-02-01 00:00:00+00:00,1444,611,609,635,0.423130,0.421745,0.439751
2,2025-03-01 00:00:00+00:00,1636,677,705,690,0.413814,0.430929,0.421760
3,2025-04-01 00:00:00+00:00,1606,680,697,663,0.423412,0.433998,0.412827
4,2025-05-01 00:00:00+00:00,1687,695,676,706,0.411974,0.400711,0.418494



La retención de usuarios se presenta muy estable, lo que sugiere que se trata de usuarios muy 
comprometidos y fieles a la empresa y los productos. 
Esto es común a todas las cohortes revisadas, lo que indica estabilidad a lo largo del periodo revisado, es 
decir, no hay caída fuerte, cohortes débiles ni deterioro en el tiempo.


---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** ...
   - **H₁ (Hipótesis alternativa):** ...
   
**Test estadístico:** ...  
**Nivel de significancia alpha:** ...

In [57]:
# tu código aquí
from statsmodels.stats.proportion import proportions_ztest
experimento = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')
experimento.head()

,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


In [58]:
# Tasa de conversión por grupo
# (No hay una diferencia estadísticamente significativa)
experimento.groupby('variante')['convirtio'].mean()


variante
control        0.156898
tratamiento    0.162860
Name: convirtio, dtype: float64

In [59]:
# Revisar tamaños de muestra

experimento.groupby('variante')['convirtio'].count()


variante
control        4965
tratamiento    5035
Name: convirtio, dtype: int64

In [60]:
##### Z-Test de proporciones 

conversiones = experimento.groupby('variante')['convirtio'].sum()
n = experimento.groupby('variante')['convirtio'].count()
stat, p_value = proportions_ztest(conversiones, n)
p_value

0.41605851639119995


alpha = 0.05
Si p-value < 0.05 Hay evidencia en contra de la Hipótesis nula (H₀) y se rechaza.
Si p-value >= 0.05 no hay evidencia suficiente contra la hipótesis nula. 
No se rechaza la Hipótesis nula, esto significa que el aumento observado puede deberse al azar, y no hay 
evidencia suficiente de un impacto real de los cambios hechos. Por el momento, no se recomienda implementar 
el cambio, pues la diferencia no es significativa. Se sugiere experimentar por más tiempo y con más usuarios.

---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---